# Day 4: Joining and Merging DataFrames - EXERCISES

## Learning Objectives
- Use merge() to combine DataFrames
- Understand different join types (inner, left, right, outer)
- Join on single and multiple columns
- Use concat() to stack DataFrames

---

## 1. Basic Merge (Inner Join)

In [2]:
# Demo: The Problem - Data in Separate Tables
import pandas as pd

# Sales transactions - only has IDs
sales = pd.DataFrame({
    'TransactionID': ['TXN-001', 'TXN-002', 'TXN-003', 'TXN-004'],
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-001', 'PROD-003'],
    'Revenue': [1500, 2000, 1800, 2500]
})

# Product master - has names
products = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-003'],
    'ProductName': ['Suite A', 'Suite B', 'Analytics'],
    'Category': ['Premium', 'Standard', 'Premium']
})

print("Sales:")
print(sales)
print("\nProducts:")
print(products)

Sales:
  TransactionID ProductID  Revenue
0       TXN-001  PROD-001     1500
1       TXN-002  PROD-002     2000
2       TXN-003  PROD-001     1800
3       TXN-004  PROD-003     2500

Products:
  ProductID ProductName  Category
0  PROD-001     Suite A   Premium
1  PROD-002     Suite B  Standard
2  PROD-003   Analytics   Premium


In [3]:
# Demo: The Solution - merge()
# Join sales with products using ProductID as the matching key

sales_with_names = sales.merge(products, on='ProductID')

print("After joining:")
print(sales_with_names)

After joining:
  TransactionID ProductID  Revenue ProductName  Category
0       TXN-001  PROD-001     1500     Suite A   Premium
1       TXN-002  PROD-002     2000     Suite B  Standard
2       TXN-003  PROD-001     1800     Suite A   Premium
3       TXN-004  PROD-003     2500   Analytics   Premium


### Exercise 1: Inner Join Practice

In [9]:
# Exercise 1: Inner Join Practice
# Task: Join orders with customers and find top customer

# Order data
orders = pd.DataFrame({
    'OrderID': ['ORD-001', 'ORD-002', 'ORD-003', 'ORD-004'],
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-A', 'CUST-C'],
    'Amount': [1500, 2000, 1800, 2500]
})

# Customer data
customers = pd.DataFrame({
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-C'],
    'CustomerName': ['Acme Corp', 'TechStart', 'Global Industries'],
    'Region': ['EMEA', 'AMER', 'APAC']
})

# Your code here:
# Step 1: Join the data on CustomerID
joined_table = pd.merge(orders, customers, on='CustomerID')

# Step 2: Find customer with highest total
# Hint: Group by CustomerName, sum Amount, use .idxmax()
customer_with_highest_total = joined_table.groupby('CustomerName')["Amount"].sum().idxmax()

# Expanded out:
grouped = joined_table.groupby('CustomerName') # Makes the customer name the index
sum_of_amount = grouped["Amount"].sum() # Get the sum for the amount
# Now to get the index with the highest value, we take the .idxmax()
customer_with_highest_total = sum_of_amount.idxmax()
print("Customer with highest total: ", customer_with_highest_total)

Customer with highest total:  Acme Corp


---
## 2. Left Join

In [ ]:
# Demo: Left Join
# Left join keeps ALL rows from the LEFT table, even if no match in right

# Sales includes PROD-999 which doesn't exist in products!
sales = pd.DataFrame({
    'TransactionID': ['TXN-001', 'TXN-002', 'TXN-003', 'TXN-004'],
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-001', 'PROD-999'],
    'Revenue': [1500, 2000, 1800, 2500]
})

products = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-003'],
    'ProductName': ['Suite A', 'Suite B', 'Analytics']
})

# Left join - keep ALL sales
result = sales.merge(products, on='ProductID', how='left')

print("Left join result:")
print(result)
print("\nNote: TXN-004 is kept, but ProductName is NaN")

### Exercise 2: Left Join for Data Quality

In [14]:
# Exercise 2: Left Join for Data Quality
# Task: Find orders that reference non-existent customers

# Data with an orphan order (CUST-Z doesn't exist)
orders = pd.DataFrame({
    'OrderID': ['ORD-001', 'ORD-002', 'ORD-003', 'ORD-004', 'ORD-005'],
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-A', 'CUST-C', 'CUST-Z'],
    'Amount': [1500, 2000, 1800, 2500, 3000]
})

customers = pd.DataFrame({
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-C'],
    'CustomerName': ['Acme Corp', 'TechStart', 'Global Industries']
})

# Your code here:
# Step 1: Left join to keep ALL orders
customers_and_orders = pd.merge(orders, customers, on='CustomerID', how='left')
print(customers_and_orders)

# Step 2: Find orphan orders where CustomerName is null
# Hint: Use .isna() to find NaN values
missing_item_index = customers_and_orders["CustomerName"].isna()
missing_item_records = customers_and_orders[missing_item_index]
print(missing_item_records)

# Step 3: Calculate total revenue from orphan orders
print("Sum of transactions with missing customer: $", missing_item_records["Amount"].sum())

   OrderID CustomerID  Amount       CustomerName
0  ORD-001     CUST-A    1500          Acme Corp
1  ORD-002     CUST-B    2000          TechStart
2  ORD-003     CUST-A    1800          Acme Corp
3  ORD-004     CUST-C    2500  Global Industries
4  ORD-005     CUST-Z    3000                NaN
   OrderID CustomerID  Amount CustomerName
4  ORD-005     CUST-Z    3000          NaN
Sum of transactions with missing customer: $ 3000


---
## 3. Outer Join

In [ ]:
# Demo: Outer Join with Indicator
# Outer join keeps EVERYTHING from BOTH tables
# indicator=True shows where each row came from

sales = pd.DataFrame({
    'TransactionID': ['TXN-001', 'TXN-002', 'TXN-003', 'TXN-004'],
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-001', 'PROD-999'],
    'Revenue': [1500, 2000, 1800, 2500]
})

products = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-003'],
    'ProductName': ['Suite A', 'Suite B', 'Analytics']
})

# Outer join with indicator
result = sales.merge(products, on='ProductID', how='outer', indicator=True)

print("Outer join with indicator:")
print(result)

print("\n_merge values:")
print(result['_merge'].value_counts())

---
## 4. Multi-Column Join

In [ ]:
# Demo: Joining on Multiple Columns
# Sometimes you need to match on more than one column

sales = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-001', 'PROD-002'],
    'Region': ['EMEA', 'AMER', 'EMEA'],
    'Units': [100, 150, 200]
})

# Pricing varies by product AND region
pricing = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-001', 'PROD-002', 'PROD-002'],
    'Region': ['EMEA', 'AMER', 'EMEA', 'AMER'],
    'Price': [199, 189, 99, 94]
})

# Join on BOTH ProductID AND Region
result = sales.merge(pricing, on=['ProductID', 'Region'])
result['Revenue'] = result['Units'] * result['Price']

print("Sales with regional pricing:")
print(result)

### Exercise 3: Multi-Column Join

In [ ]:
# Exercise 3: Multi-Column Join
# Task: Join sales with regional targets and calculate achievement

sales = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'AMER', 'AMER', 'APAC'],
    'Quarter': ['Q1', 'Q2', 'Q1', 'Q2', 'Q1'],
    'Revenue': [125000, 138000, 98000, 105000, 67000]
})

targets = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'AMER', 'AMER', 'APAC', 'APAC'],
    'Quarter': ['Q1', 'Q2', 'Q1', 'Q2', 'Q1', 'Q2'],
    'Target': [120000, 130000, 100000, 100000, 70000, 75000]
})

# This could cause issues:
#incorrect_example = pd.merge(sales,targets, on='Region')
#print(incorrect_example)

# Your code here:
# Step 1: Join on BOTH Region AND Quarter
# Hint: Use on=['Region', 'Quarter']
joined_table = pd.merge(sales,targets, on=['Region', 'Quarter'])
#print(joined_table)

# Step 2: Calculate achievement percentage
# Formula: (Revenue / Target) * 100, round to 1 decimal
joined_table["Achievement"] = (joined_table["Revenue"] / joined_table["Target"] * 100).round(1)

# Step 3: Find region-quarters that exceeded target (>= 100%)
print("High achievement: ", joined_table[joined_table["Achievement"] >= 100])

  Region Quarter  Revenue  Target  Achievement
0   EMEA      Q1   125000  120000        104.2
1   EMEA      Q2   138000  130000        106.2
3   AMER      Q2   105000  100000        105.0


---
## 5. Concatenating DataFrames

In [ ]:
# Demo: Concatenate Rows (Stacking)
# concat() stacks DataFrames vertically (adds rows)
# Use for combining same-structure data from different periods

jan_sales = pd.DataFrame({
    'Product': ['Suite A', 'Suite B'],
    'Revenue': [15000, 12000],
    'Month': ['Jan', 'Jan']
})

feb_sales = pd.DataFrame({
    'Product': ['Suite A', 'Suite B'],
    'Revenue': [18000, 13500],
    'Month': ['Feb', 'Feb']
})

# Stack them vertically
all_sales = pd.concat([jan_sales, feb_sales], ignore_index=True)

print("Combined sales:")
print(all_sales)

### Exercise 4: Concat and Merge

In [ ]:
# Exercise 4: Concat then Merge
# Task: Combine quarterly files, then add product info

q1_sales = pd.DataFrame({
    'Product': ['Suite A', 'Suite B', 'Analytics'],
    'Revenue': [45000, 32000, 38000],
    'Quarter': ['Q1', 'Q1', 'Q1']
})

q2_sales = pd.DataFrame({
    'Product': ['Suite A', 'Suite B', 'Analytics'],
    'Revenue': [52000, 38000, 42000],
    'Quarter': ['Q2', 'Q2', 'Q2']
})

products = pd.DataFrame({
    'Product': ['Suite A', 'Suite B', 'Analytics'],
    'Category': ['Premium', 'Standard', 'Premium']
})

# Your code here:
# Step 1: Combine quarterly files with concat()
# Hint: Use ignore_index=True


# Step 2: Merge with products to add Category


# Step 3: Calculate total by Category

---
## Challenge Exercise: Build Integrated Dataset

In [ ]:
# Challenge: Build Integrated Dataset
# Task: Join three tables and calculate profit by Region and Category

# Transaction data
transactions = pd.DataFrame({
    'TransactionID': ['TXN-001', 'TXN-002', 'TXN-003', 'TXN-004', 'TXN-005'],
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-A', 'CUST-C', 'CUST-B'],
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-001', 'PROD-003', 'PROD-002'],
    'Amount': [1500, 2000, 1800, 2500, 1900]
})

# Customer data
customers = pd.DataFrame({
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-C'],
    'CustomerName': ['Acme Corp', 'TechStart', 'Global Ind'],
    'Region': ['EMEA', 'AMER', 'APAC']
})

# Product data
products = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-003'],
    'ProductName': ['Suite A', 'Suite B', 'Analytics'],
    'Category': ['Premium', 'Standard', 'Premium'],
    'Cost': [800, 1100, 1300]
})

# Your code here:
# Step 1: Join transactions with customers


# Step 2: Join with products


# Step 3: Calculate profit (Amount - Cost)


# Step 4: Create profit summary by Region and Category


# Step 5: Find most profitable combination

---
## Summary

**You've learned:**
- merge() for combining DataFrames on common columns
- Join types: inner (default), left, right, outer
- Joining on multiple columns
- Using indicator to track merge results
- concat() for stacking DataFrames

**Key Takeaways:**
- Left join is most common (keep all your data)
- Use indicator=True to find data quality issues
- concat() for same-structure stacking, merge() for joining

**Next:** Data visualization!